In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
#Hyperparameters
INPUT_CHANNELS = 3  # RGB images
OUTPUT_SIZE = 5  # Number of actions the pacman can take

In [3]:
class CNN(nn.Module):
    def __init__(self, input_channels, output_size): 
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(INPUT_CHANNELS, 32, kernel_size=4, stride=4, padding=1) 
        self.pool = nn.MaxPool2d(kernel_size=4, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=1)
        self.fc1 = nn.Linear(384, OUTPUT_SIZE)                                          
       
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        
        return x
    
    #Nouvau CNN

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [5]:
class REINFORCE(torch.nn.Module):

    def __init__(self, ):
        super(REINFORCE, self).__init__()

        self.learning_rate = 1e-4 #correspond au alpha dans l'equation de mise a jour des poids
        self.gamma = 0.99   #Discount factor
        self.eps = 0.7  # small number for mathematical stability and 1 for only random actions

        self.probs = []  # Stores probability values of the sampled action
        self.rewards = []  # Stores the corresponding rewards

        self.model = CNN(INPUT_CHANNELS, OUTPUT_SIZE).to(device) # 3 input channels for RGB image, 5 output size for 5 first actions 
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate) #Adam optimizer pour mettre a jour les poids

        

        
    def select_action(self, observation):
        
        observation = torch.tensor(observation).to(device)
        
        #new softmax with temperature
        temperature = 1.5 #if temperature = 1, it's the same as the old softmax, if t > 1, it's more random, if t < 1, it's less random
        logits = self.model(observation)
        scaled_logits = logits / temperature

        action_probs = F.softmax(scaled_logits, dim=1)
        # Add noise to the action probabilities
        #noise = torch.rand_like(action_probs)
        #action_probs = (action_probs + 0.2*noise) / 2

        #ancien softmax
        #action_probs = F.softmax(self.model(observation), dim=1)
        
        #print(action_probs)

        # Decay the epsilon value : epsilon high au debut puis baisse pour avoir moins de random actions
        if self.eps > 0.1:
            self.eps = self.eps - 0.000005 # linear decraese
            #print("epsilon : {}".format(self.eps))
        
        #------------- epsilon-greedy policy ----------------
        # Choose the best action with probability 1 - eps, random action with probability eps
        if random.random() > self.eps:
            #action = torch.argmax(action_probs)
            action = torch.multinomial(action_probs, 1)
        else:
            # select random action
            action = torch.randint(0, OUTPUT_SIZE, (1,))
        
        self.probs.append(action_probs[0, action])
        #print(f"action is {action}")
        return action.item()


    def update(self):
        running_g = 0
        gs = []

        # Discounted return (backwards) - [::-1] will return an array in reverse
        for R in self.rewards[::-1]:
            running_g = R + self.gamma * running_g
            gs.insert(0, running_g)

        deltas = torch.tensor(gs)

        loss = 0 
        entropy = 0

        # minimize -1 * prob * reward obtained
        for log_prob, delta in zip(self.probs, deltas):
            loss += log_prob.mean() * delta * (-1)
            entropy -= log_prob.mean() * torch.log(log_prob.mean())

        loss -= 0.001 * entropy  # Add entropy regularization term : plus eleve = plus de random actions

        # Update the policy network
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Empty / zero out all episode-centric/related variables
        self.probs = []
        self.rewards = []

    def load_state_dict(self, state_dict):
       super(REINFORCE, self).load_state_dict(state_dict)


In [6]:
def preprocess_observation(observation):
  
    observation_array = np.array(observation[0]) 
    
    #crop the obsservation height 210 -> 160
    observation = observation[:172, :, :]

    # take only 1/2 pixels of observation
    observation = observation[::2, ::2, :]
    
    # Convert the observation to grayscale
    #observation = np.mean(observation, axis=2)
    # Normalize the observation to the range [0, 1] so that it is accepted by the CNN
    observation = observation / 255.0

    # Resize the observation to the expected input size of the CNN
    observation = torch.tensor(observation, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
    
    return observation

In [8]:
def pacman_train():
    # Load the game environment
    env = gym.make('ALE/MsPacman-v5') 
    agent = REINFORCE()
    total_reward_tab = []
    true_total_reward_tab = []

    Samples_true_rewards_plot = []
    nbr_samples = 0

    num_episodes = 10001
    steps_done = 0
    
    
    # Load the agent state if a saved state exists
    #try:
        #agent.load_state_dict(torch.load(f"reinforce_episode_1000copie.pth "))
    #except FileNotFoundError:
        #pass
        

    for i_episode in range(num_episodes):
        observation, game_state_info = env.reset()

        done = False
        total_reward = 0
        true_total_reward = 0
        previous_action_reward = 0
        lives = game_state_info['lives']
                
        while not done:
            observation = preprocess_observation(observation)

            ##### to see the preprocessed observations given
            #image = observation[0]
            #image = image.permute(1, 2, 0)
            # # Plot the image
            #plt.imshow(image)
            #plt.show()

            #print(f"\nObservation est :\n{observation}")
            action = agent.select_action(observation)

            # These represent the next observation, the reward from the step,
            # if the episode is terminated, if the episode is truncated and
            # additional info from the step
            observation, reward, terminated, truncated, game_state_info = env.step(action)
            true_total_reward += reward

            # If the number of lives has decreased, subtract 50 from the reward
            if game_state_info['lives'] < lives:
                reward -= 50
                lives = game_state_info['lives']
                
            elif reward == previous_action_reward:
                reward -= 1 # si il bouge pas ou qu'il fait pas un bon mouv -1
            elif reward > 10:
                reward = 10
            
            agent.rewards.append(reward)

            # End the episode when either truncated or terminated is true
            #  - truncated: The episode duration reaches max number of timesteps
            #  - terminated: Any of the state space values is no longer finite.

            done = terminated or truncated
            previous_action_reward = reward

            if steps_done % 25 == 0:
                agent.update()
            
            steps_done += 1
            total_reward += reward

        print(f'Episode: {i_episode}, True game Total reward: {true_total_reward}')
        total_reward_tab.append(total_reward)
        true_total_reward_tab.append(true_total_reward)

        if i_episode % 50 == 0:
            torch.save(agent.state_dict(), f"pth_de_pacman_reinforce_multinomial_eps01/reinforce_episode_{i_episode}.pth")
            Samples_true_rewards_plot.append(true_total_reward)
            nbr_samples += 1
            
        if i_episode % 100 == 0:
            #plot sliding average of the total rewards
            plt.plot(pd.Series(true_total_reward_tab).rolling(30).mean(), linewidth=0.5, color='red')
            plt.xlabel('Episode')
            plt.ylabel('Total Reward')
            plt.title('Number of episodes {} sliding average'.format(i_episode))
            plt.savefig(f"pth_de_pacman_reinforce_multinomial_eps01/sliding_average_episode_{i_episode}.png")

            plt.show()

    env.close()

    # After all episodes are done, plot the total rewards
    plt.scatter(range(num_episodes), true_total_reward_tab, s = 0.1, color='blue')
    # plot a tendancy line to see the evolution of the total reward
    episodes_idx = np.arange(num_episodes)
    sns.regplot(x=episodes_idx, y=true_total_reward_tab, order=1, color='orange')
    
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('Number of episodes {}'.format(num_episodes))
    plt.savefig("pth_de_pacman_reinforce_multinomial_eps01/total_reward_reinforce_.png")

    plt.show()

    #plot sliding average of the total rewards
    plt.plot(pd.Series(true_total_reward_tab).rolling(30).mean(), linewidth=0.5, color='red')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('Number of episodes {} sliding average'.format(num_episodes))
    plt.savefig("pth_de_pacman_reinforce_multinomial_eps01/sliding_average_reward_reinforce.png")

    plt.show()


    # Plot the rewards obtained every 50 episodes
    plt.plot(Samples_true_rewards_plot, linewidth=0.5, color='red')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('Number of episodes {}'.format(num_episodes))
    plt.savefig("pth_de_pacman_reinforce_multinomial_eps01/total_reward_reinforce_50.png")
    
    plt.show()




In [9]:
def pacman_play():
    # Load the game environment
    env = gym.make('ALE/MsPacman-v5', render_mode='human')  

    agent = REINFORCE()
    try:
        agent.load_state_dict(torch.load(f"agent.pth"))
    except FileNotFoundError:
        pass
    
    # Initialize the CNN
    cnn = CNN(INPUT_CHANNELS, OUTPUT_SIZE)

    observation, game_state_info = env.reset()  # Reset the environment for a new episode : ressort le tableau observation + game state info
    done = False
    score = 0

    while not done:

        observation = preprocess_observation(observation)
        action = agent.select_action(observation)
        observation, reward, terminated, truncated, game_state_info = env.step(action)
        score += reward  # Update the score
        done = terminated or truncated
        
     
    env.close()  # Close the environment
    print(f"c'est bon c'est fini")



In [10]:
if __name__ == '__main__':
    #pacman_play()
    pacman_train()
    



A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
ipykernel_21152/1370307875.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  observation = torch.tensor(observation).to(device)


: 